In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import random
import networkx as nx
from collections import defaultdict
import itertools

In [2]:
# ── Cache parameters — must match hm_foaf_experiment_sampled.ipynb ───────────
TARGET_USERS           = 1000
MIN_TRAIN_INTERACTIONS = 20
SEED                   = 42
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cached files not found in '{SAMPLED_DATA_DIR}/'.\n"
    "Run the hm_foaf_experiment_sampled.ipynb preprocessing section first "
    "to generate the cache."
)
print(f"Cache tag: {cache_tag}")

Cache tag: u1000_m20_s42


In [3]:
# ── Load sampled H&M dataset from cache ──────────────────────────────────────
train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded from cache: {cache_tag}")
print(f"Total Users: {n_users}")
print(f"Total Items: {n_items}")
train_df.head()

Loaded from cache: u1000_m20_s42
Total Users: 948
Total Items: 7418


,customer_id,product_code,bought
0,115,339,4
1,634,546,1
2,630,230,2
3,233,222,2
4,778,9,1


In [4]:
# ── Build per-user train/test dicts for Gossip Learning ──────────────────────
# val is available but unused here; merge into train if desired.
def df_to_dict(df):
    # Rename to canonical (user_id, item_id, rating) regardless of source column names
    df = df.copy()
    df.columns = ["user_id", "item_id", "rating"] + list(df.columns[3:])
    d = {}
    for uid, grp in df.groupby("user_id"):
        d[int(uid)] = grp[["user_id", "item_id", "rating"]].values
    return d

train_data_dict = df_to_dict(train_df)
test_data_dict  = df_to_dict(test_df)

num_users = n_users
num_items = n_items

# Only keep users that appear in BOTH train and test
common_users = set(train_data_dict.keys()) & set(test_data_dict.keys())
train_data_dict = {u: v for u, v in train_data_dict.items() if u in common_users}
test_data_dict  = {u: v for u, v in test_data_dict.items()  if u in common_users}

total_train = sum(len(v) for v in train_data_dict.values())
total_test  = sum(len(v) for v in test_data_dict.values())
print(f"Active users (train∩test): {len(common_users)}")
print(f"Train interactions: {total_train}")
print(f"Test  interactions: {total_test}")
print(f"Density: {(total_train + total_test) / (num_users * num_items) * 100:.4f}%")

Active users (train∩test): 945
Train interactions: 38557
Test  interactions: 15696
Density: 0.7715%


In [5]:
# ── Communication graph ───────────────────────────────────────────────────────
# Build graph over the actual active user IDs (not range(num_users)),
# so that graph node labels match train_data_dict keys.
active_ids = sorted(common_users)
G = nx.erdos_renyi_graph(len(active_ids), 0.05, seed=SEED)
# Relabel nodes from 0..N-1 to the actual user IDs
G = nx.relabel_nodes(G, {i: uid for i, uid in enumerate(active_ids)})
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Avg degree: {np.mean([d for _, d in G.degree()]):.2f}")

Graph: 945 nodes, 22097 edges
Avg degree: 46.77


In [6]:
# ── Model & helper definitions (unchanged from ML-100K template) ──────────────

def initX(k, R_min=0.0, R_max=1.0):
    """Initialize a single user embedding + bias.
    Range adjusted to [0,1] for implicit binary ratings."""
    scale = np.sqrt((R_max - R_min) / k)
    x = torch.rand(k) * scale
    b = torch.tensor(R_min / 2.0)
    return x, b

def initY(num_items, k, R_min=0.0, R_max=1.0):
    Y = torch.zeros(num_items, k)
    c = torch.zeros(num_items)
    t = torch.zeros(num_items)
    for j in range(num_items):
        Y[j], c[j] = initX(k, R_min, R_max)
    return Y, c, t


class RatingDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user, item, rating = self.data[idx]
        return user, item, rating.float()


class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=20):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.bias = nn.Parameter(torch.zeros(num_items))

    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        return (u * i).sum(1) + self.bias[items]


def compress_model(Y, c, t, compression_ratio):
    num_items = Y.shape[0]
    k = max(1, int(compression_ratio * num_items))
    indices = np.random.choice(num_items, k, replace=False)
    return indices, Y[indices], c[indices], t[indices]


def merge_model(Y, c, t, Y_recv, c_recv, t_recv, indices):
    for i, j in enumerate(indices):
        if t_recv[i] > 0:
            tj, t_new = t[j].item(), t_recv[i].item()
            w = t_new / (tj + t_new) if (tj + t_new) > 0 else 0.5
            t[j] = max(tj, t_new)
            Y[j] = (1 - w) * Y[j] + w * Y_recv[i]
            c[j] = (1 - w) * c[j] + w * c_recv[i]
    return Y, c, t


def train_local(model, dataloader, optimizer, criterion, device):
    model.train()
    for users, items, ratings in dataloader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        preds = model(users, items)
        loss = criterion(preds, ratings)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


def evaluate(model, dataloader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for users, items, ratings in dataloader:
            users, items = users.to(device), items.to(device)
            pred = model(users, items)
            preds.extend(pred.cpu().numpy())
            trues.extend(ratings.numpy())
    return np.sqrt(np.mean((np.array(preds) - np.array(trues))**2))


def gossip_learning(num_users, num_items, train_data_dict, test_data_dict,
                    graph, epochs=10, emb_dim=20, lr=0.01, compression_ratio=0.1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    user_models, user_Y, user_c, user_t = {}, {}, {}, {}
    train_loaders, test_loaders, optimizers = {}, {}, {}
    criterion = nn.MSELoss()

    for u in graph.nodes():
        if u not in train_data_dict:
            continue
        model = MF(num_users, num_items, emb_dim).to(device)
        user_models[u] = model
        optimizers[u] = optim.Adam(model.parameters(), lr=lr)
        Y, c, t = initY(num_items, emb_dim)
        user_Y[u] = Y.clone()
        user_c[u] = c.clone()
        user_t[u] = t.clone()
        model.item_emb.weight.data = Y.clone()
        model.bias.data = c.clone()
        train_loaders[u] = DataLoader(RatingDataset(train_data_dict[u]),
                                      batch_size=64, shuffle=True)
        test_loaders[u]  = DataLoader(RatingDataset(test_data_dict[u]),
                                      batch_size=256)

    active = list(user_models.keys())

    for epoch in range(epochs):
        for u in active:
            train_local(user_models[u], train_loaders[u], optimizers[u], criterion, device)

        for u in active:
            neighbors = [v for v in graph.neighbors(u) if v in user_models]
            if not neighbors:
                continue
            partner = random.choice(neighbors)
            indices, Y_s, c_s, t_s = compress_model(
                user_Y[u], user_c[u], user_t[u], compression_ratio)
            user_Y[partner], user_c[partner], user_t[partner] = merge_model(
                user_Y[partner], user_c[partner], user_t[partner],
                Y_s, c_s, t_s, indices)
            user_models[partner].item_emb.weight.data = user_Y[partner]
            user_models[partner].bias.data = user_c[partner]

    rmses = [evaluate(user_models[u], test_loaders[u], device) for u in active]
    return np.mean(rmses)

In [7]:
# ── Parameter sweep ───────────────────────────────────────────────────────────
param_grid = {
    "emb_dim":           [10, 20, 50, 70],
    "lr":                [0.01, 0.005, 0.001],
    "compression_ratio": [0.3, 0.1],
    "epochs":            [100],
}

results = []

for emb_dim, lr, cr, epochs in itertools.product(
        param_grid["emb_dim"], param_grid["lr"],
        param_grid["compression_ratio"], param_grid["epochs"]):
    print(f"\nConfig: emb_dim={emb_dim}, lr={lr}, compression={cr}, epochs={epochs}")
    rmse = gossip_learning(
        num_users, num_items,
        train_data_dict, test_data_dict,
        G,
        epochs=epochs, emb_dim=emb_dim, lr=lr, compression_ratio=cr)
    print(f"Test RMSE: {rmse:.4f}")
    results.append([emb_dim, lr, cr, epochs, rmse])


Config: emb_dim=10, lr=0.01, compression=0.3, epochs=100
Test RMSE: 1.4327

Config: emb_dim=10, lr=0.01, compression=0.1, epochs=100
Test RMSE: 1.4481

Config: emb_dim=10, lr=0.005, compression=0.3, epochs=100
Test RMSE: 1.4334

Config: emb_dim=10, lr=0.005, compression=0.1, epochs=100
Test RMSE: 1.4232

Config: emb_dim=10, lr=0.001, compression=0.3, epochs=100
Test RMSE: 1.5456

Config: emb_dim=10, lr=0.001, compression=0.1, epochs=100
Test RMSE: 1.5381

Config: emb_dim=20, lr=0.01, compression=0.3, epochs=100
Test RMSE: 1.4656

Config: emb_dim=20, lr=0.01, compression=0.1, epochs=100
Test RMSE: 1.4569

Config: emb_dim=20, lr=0.005, compression=0.3, epochs=100
Test RMSE: 1.4715

Config: emb_dim=20, lr=0.005, compression=0.1, epochs=100
Test RMSE: 1.4579

Config: emb_dim=20, lr=0.001, compression=0.3, epochs=100
Test RMSE: 1.5030

Config: emb_dim=20, lr=0.001, compression=0.1, epochs=100
Test RMSE: 1.5016

Config: emb_dim=50, lr=0.01, compression=0.3, epochs=100
Test RMSE: 1.4962

Con

In [8]:
# ── Results table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(
    results,
    columns=["embedding_dim", "learning_rate", "compression_ratio", "epochs", "test_rmse"])

# Optionally persist
# results_df.to_csv(HM_DIR / "gossip_tuning_results_hm.csv", index=False)

print("Best config:")
print(results_df.loc[results_df["test_rmse"].idxmin()])
results_df

Best config:
embedding_dim         10.000000
learning_rate          0.005000
compression_ratio      0.100000
epochs               100.000000
test_rmse              1.423198
Name: 3, dtype: float64


,embedding_dim,learning_rate,compression_ratio,epochs,test_rmse
0,10,0.010,0.3,100,1.432736
1,10,0.010,0.1,100,1.448094
2,10,0.005,0.3,100,1.433371
3,10,0.005,0.1,100,1.423198
4,10,0.001,0.3,100,1.545632
5,10,0.001,0.1,100,1.538123
6,20,0.010,0.3,100,1.465556
7,20,0.010,0.1,100,1.456858
8,20,0.005,0.3,100,1.471528
9,20,0.005,0.1,100,1.457917


In [9]:
min(results_df["test_rmse"])

1.4231979846954346